# 🌦️ XGBoost Weather Forecasting & Prediction Model
### **Dataset**: `bengaluru_preprocessed.csv` (2001 - 2026)
### **Target Variable**: Hourly 2-Meter Air Temperature (`temperature_2m`)

---
### 📌 Overview & Missing Elements Addressed from Preprocessing Notebook:
1. **Proper Time-Series Handling & Lookahead Prevention**: Avoided random shuffling (`train_test_split(shuffle=False)`) which leaks future data into the past. We implement chronological splitting (Train: 2001–2021, Val: 2021–2024, Test: 2024–2026).
2. **Categorical Feature Encoding**: The previous preprocessing left `season` and `day_name` as raw strings in the CSV; here we apply One-Hot Encoding (`pd.get_dummies`).
3. **Cyclical Temporal Transformations**: Linear integer representations of `hour` (0-23) and `month` (1-12) miss cyclical continuity (e.g. 23:00 is right next to 00:00). We encode them with sine and cosine periodic transformations.
4. **Lag & Rolling Momentum Features**: Weather has strong thermal inertia. We engineer 1-hour and 24-hour lag features, plus 24-hour rolling averages and standard deviations.
5. **Collinear Target Leakage Prevention**: Features like simultaneous `apparent_temperature`, `dew_point_2m`, and surface soil temperatures that would cause artificial leakages are properly handled.
6. **Comprehensive Multi-Metric Evaluation**: R², RMSE, MAE, MSE, MAPE, Explained Variance, Residual diagnostics, Feature Importance, and Model Serialization (`joblib` & `JSON`).

## 📦 Step 1: Import Required Libraries

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import xgboost as xgb
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error,
    explained_variance_score
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120

print(f"XGBoost Version : {xgb.__version__}")
print(f"Pandas Version  : {pd.__version__}")

## 📂 Step 2: Load the Preprocessed Dataset (`bengaluru_preprocessed.csv`)

In [ ]:
df = pd.read_csv('bengaluru_preprocessed.csv')
print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

## ⚙️ Step 3: Advanced Feature Engineering & Addressing Missed Steps
- **DateTime Conversion & Sorting**
- **Cyclical Trigonometric Transformations** (`hour_sin`, `hour_cos`, `month_sin`, `month_cos`, `doy_sin`, `doy_cos`)
- **Lag Features** (1-hour lag, 24-hour lag)
- **Rolling Window Statistics** (24-hour mean, 24-hour standard deviation)
- **One-Hot Encoding** (`season`, `day_name`)

In [ ]:
# 1. Datetime formatting and chronological ordering
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

# 2. Cyclical Temporal Encoding
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)
df['day_of_year'] = df['time'].dt.dayofyear
df['doy_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)

# 3. Lag Features for thermal inertia
lag_features = ['temperature_2m', 'relative_humidity_2m', 'surface_pressure', 'wind_speed_10m']
for col in lag_features:
    if col in df.columns:
        df[f'{col}_lag_1h'] = df[col].shift(1)
        df[f'{col}_lag_24h'] = df[col].shift(24)

# 4. Rolling Window Statistics
df['temp_rolling_mean_24h'] = df['temperature_2m'].shift(1).rolling(window=24).mean()
df['temp_rolling_std_24h'] = df['temperature_2m'].shift(1).rolling(window=24).std()
df['humidity_rolling_mean_24h'] = df['relative_humidity_2m'].shift(1).rolling(window=24).mean()

# Drop warm-up NaN rows caused by 24h lag/rolling
df = df.dropna().reset_index(drop=True)

# 5. One-Hot Encoding for remaining categorical columns
cat_cols = [c for c in ['season', 'day_name'] if c in df.columns]
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=float)

print(f"Final Engineered Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
df[['time', 'temperature_2m', 'hour_sin', 'hour_cos', 'temperature_2m_lag_1h', 'temp_rolling_mean_24h']].head()

## ✂️ Step 4: Time-Series Aware Chronological Data Splitting
- **Training Set (80%)**: 2001 to mid-2021
- **Validation Set (10%)**: mid-2021 to early-2024 (Used for Early Stopping)
- **Test Set (10%)**: early-2024 to late-2026 (Unseen Out-of-Time Test Benchmark)

In [ ]:
target_col = 'temperature_2m'
drop_cols = ['time', target_col, 'apparent_temperature', 'dew_point_2m', 'soil_temperature_0_to_7cm']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols]
y = df[target_col]
timestamps = df['time']

n = len(df)
train_end = int(n * 0.8)
val_end = int(n * 0.9)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

print(f"Training Set   : {len(X_train):,} samples ({timestamps.iloc[0].date()} to {timestamps.iloc[train_end - 1].date()})")
print(f"Validation Set : {len(X_val):,} samples ({timestamps.iloc[train_end].date()} to {timestamps.iloc[val_end - 1].date()})")
print(f"Test Set       : {len(X_test):,} samples ({timestamps.iloc[val_end].date()} to {timestamps.iloc[-1].date()})")
print(f"Feature Count  : {X.shape[1]} features")

## 🚀 Step 5: XGBoost Model Architecture & Training with Early Stopping

In [ ]:
params = {
    'n_estimators': 1000,
    'learning_rate': 0.03,
    'max_depth': 6,
    'min_child_weight': 3,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'gamma': 0.1,
    'reg_alpha': 0.05,
    'reg_lambda': 1.0,
    'random_state': 42,
    'n_jobs': -1,
    'tree_method': 'hist',
    'eval_metric': 'rmse',
    'early_stopping_rounds': 50
}

model = xgb.XGBRegressor(**params)

print("Training XGBoost Regressor...")
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=100
)

print(f"\nOptimal Boosting Iterations: {model.best_iteration}")
print(f"Best Validation RMSE: {model.best_score:.4f} °C")

## 📊 Step 6: Model Testing & Comprehensive Evaluation Metrics

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

def evaluate_performance(y_true, y_pred, split_label):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    evs = explained_variance_score(y_true, y_pred)
    return {
        'Dataset Split': split_label,
        'RMSE (°C)': rmse,
        'MAE (°C)': mae,
        'MSE (°C²)': mse,
        'R² Score': r2,
        'MAPE (%)': mape,
        'Explained Variance': evs
    }

metrics_train = evaluate_performance(y_train, y_pred_train, 'Train Set')
metrics_test = evaluate_performance(y_test, y_pred_test, 'Test Set (Unseen)')

metrics_df = pd.DataFrame([metrics_train, metrics_test]).set_index('Dataset Split')
display(metrics_df.style.format('{:.4f}').background_gradient(cmap='Blues'))

## 📈 Step 7: Visual Diagnostics & Residual Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Loss / Learning Curves
evals_result = model.evals_result()
axes[0, 0].plot(evals_result['validation_0']['rmse'], label='Train RMSE', color='#1f77b4', lw=2)
axes[0, 0].plot(evals_result['validation_1']['rmse'], label='Validation RMSE', color='#ff7f0e', lw=2)
axes[0, 0].axvline(model.best_iteration, color='red', linestyle='--', alpha=0.8, label=f'Best Iteration ({model.best_iteration})')
axes[0, 0].set_title('Learning Curves (RMSE vs Boosting Iterations)', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].set_ylabel('RMSE (°C)')
axes[0, 0].legend()

# 2. Actual vs Predicted (Recent 300 Hours Zoom)
zoom = 300
axes[0, 1].plot(range(zoom), y_test.iloc[-zoom:].values, label='Actual Temperature', color='#2ca02c', lw=2)
axes[0, 1].plot(range(zoom), y_pred_test[-zoom:], label='XGBoost Predicted', color='#d62728', linestyle='--', lw=2)
axes[0, 1].set_title(f'Actual vs Predicted Temperature (Recent {zoom} Hours)', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Hours')
axes[0, 1].set_ylabel('Temperature (°C)')
axes[0, 1].legend()

# 3. Residual Distribution
residuals = y_test - y_pred_test
sns.histplot(residuals, kde=True, ax=axes[1, 0], color='#9467bd', bins=40)
axes[1, 0].axvline(0, color='black', linestyle='--', lw=1.5)
axes[1, 0].set_title(f'Residuals Distribution (Mean={residuals.mean():.3f}, Std={residuals.std():.3f})', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Prediction Error (°C)')
axes[1, 0].set_ylabel('Count')

# 4. Feature Importance (Top 15)
imp_df = pd.DataFrame({'Feature': feature_cols, 'Importance': model.feature_importances_}).sort_values('Importance', ascending=False).head(15)
sns.barplot(x='Importance', y='Feature', data=imp_df, ax=axes[1, 1], palette='viridis')
axes[1, 1].set_title('Top 15 Feature Importances (Gain)', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Gain Score')

plt.tight_layout()
plt.show()

## 💾 Step 8: Save Model & Metadata Artifacts

In [ ]:
# Save model via joblib and native XGBoost JSON
joblib.dump(model, 'xgboost_weather_model.joblib')
model.save_model('xgboost_weather_model.json')

metadata = {
    'model_name': 'XGBoost Regressor (Weather Forecast)',
    'target': target_col,
    'features': feature_cols,
    'test_metrics': metrics_df.loc['Test Set (Unseen)'].to_dict()
}

with open('xgboost_model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print("Artifacts successfully saved:")
print("  • xgboost_weather_model.joblib")
print("  • xgboost_weather_model.json")
print("  • xgboost_model_metadata.json")

## 🎯 Step 9: Sample Inference Demonstration on Unseen Data

In [ ]:
sample_X = X_test.iloc[:10]
sample_actual = y_test.iloc[:10].values
sample_preds = model.predict(sample_X)

demo_df = pd.DataFrame({
    'Time': timestamps.iloc[val_end:val_end+10].values,
    'Actual Temp (°C)': np.round(sample_actual, 2),
    'Predicted Temp (°C)': np.round(sample_preds, 2),
    'Absolute Error (°C)': np.round(np.abs(sample_actual - sample_preds), 2)
})
demo_df